<a href="https://colab.research.google.com/github/mirdbg/Entrega_RAG/blob/main/notebook_entrega/Entrega_RAG_v5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agente para el análisis de informes financieros 10-K
**Práctica de LLMs aplicados a Finanzas · MIAX · Entrega final documentada**

## Objetivo y filosofía de la solución

El objetivo de este trabajo es construir un **agente investigador sobre informes 10-K** capaz de responder preguntas financieras de forma trazable. El problema no consiste únicamente en obtener una respuesta plausible: el sistema debe escoger la fuente adecuada, recuperar evidencia relevante, verificar las cifras y dejar una trayectoria que permita auditar cómo ha llegado a la respuesta. Por tanto, la calidad se entiende como la combinación de **corrección, trazabilidad, robustez, coste y latencia**.

La arquitectura parte de una idea central: **el retrieval es una herramienta del agente, no la arquitectura completa**. Distintos tipos de pregunta requieren mecanismos distintos. Una cifra financiera exacta debe consultarse en XBRL; una pregunta sobre riesgos, estrategia o comentarios de la dirección requiere recuperar texto del 10-K; y la lectura de una sección completa se reserva para situaciones en las que los fragmentos recuperados no contienen contexto suficiente. Esta separación evita que el LLM utilice una fuente cómoda pero metodológicamente incorrecta.

El notebook integra en un único flujo reproducible el trabajo de las sesiones **Herramientas y bucle del agente** y **Robustez y evaluación**. Se ha reorganizado el material para que el resultado pueda leerse como una entrega completa: primero se prepara el entorno y el corpus; después se construyen las herramientas; a continuación se estudia y mejora el retrieval; finalmente se monta el agente, se incorporan guardrails y se evalúa el sistema.

### Componentes principales

1. **XBRL como fuente numérica autorizada.** Los hechos estructurados se utilizan para magnitudes financieras exactas. Esto evita extraer números de tablas partidas o de menciones narrativas potencialmente ambiguas.
2. **Retrieval denso con BGE + FAISS.** Constituye el baseline semántico y permite recuperar fragmentos por similitud de significado.
3. **Filtrado por metadatos.** Cuando conocemos empresa, ejercicio o sección, reducimos el espacio de búsqueda al documento correcto. Es especialmente importante porque muchos 10-K repiten formulaciones entre ejercicios.
4. **BM25 + búsqueda densa.** Se añade una señal léxica para consultas en las que importan términos exactos, cifras, nombres o vocabulario específico. Ambas señales se fusionan mediante Reciprocal Rank Fusion.
5. **Reescritura de consultas.** Las preguntas del usuario pueden estar en español mientras que el corpus está en inglés. El modelo transforma la pregunta en una consulta de retrieval más próxima al vocabulario del informe.
6. **Agente con herramientas.** El LLM decide qué herramienta necesita y puede encadenar varias llamadas cuando la pregunta lo requiere.
7. **Salida estructurada.** La respuesta no se limita a texto libre: incluye cifra, unidad, ticker, ejercicio, fuente, cita y `chunk_id`. Esto permite evaluar automáticamente el resultado.
8. **Guardrails.** Se limitan llamadas al modelo y herramientas, y se verifica que las cifras generadas sean compatibles con los hechos XBRL.
9. **Evaluación automática.** Se miden cita, cifra, trayectoria, recall del retrieval, coste, latencia y número de llamadas.

### Decisión de diseño: conservar `miax_s1.py` y `miax_s2.py` sin modificaciones

Los módulos auxiliares entregados para las sesiones se mantienen intactos. `miax_s1.py` representa la búsqueda densa de partida y sirve como referencia reproducible del baseline. `miax_s2.py` contiene infraestructura auxiliar —carga del índice, codificación, BM25, métricas y utilidades— que no constituye por sí misma la solución final. Las decisiones específicas del grupo quedan explícitas en este notebook, lo que facilita distinguir **qué se proporciona** y **qué se construye o modifica como parte de la práctica**.

### Reproducibilidad

El notebook está preparado para Google Colab. Los datos se leen desde Google Drive y las credenciales se obtienen mediante **Colab Secrets**, por lo que ninguna API key aparece en el código ni debe subirse al repositorio. La intención es que, con los ficheros de la práctica situados en la carpeta esperada, pueda ejecutarse de arriba abajo sin editar código.

> **Criterio de lectura.** En cada sección se explica primero qué problema se quiere resolver, por qué se adopta una determinada decisión y qué esperamos obtener. Después aparece la implementación. Así, el notebook funciona simultáneamente como código ejecutable y como memoria técnica de la solución.


## 1. Instalación y configuración del entorno

### Qué se hace en esta etapa

Antes de construir el agente necesitamos fijar un entorno de ejecución reproducible. En proyectos basados en LangChain y LangGraph esto es especialmente importante porque las APIs evolucionan con rapidez y pequeñas diferencias de versión pueden cambiar imports, middlewares o contratos de `create_agent`. Por ese motivo se fijan explícitamente las versiones principales utilizadas durante el desarrollo.

Se instalan cuatro grupos de dependencias:

- **LangChain/LangGraph**, para construir el agente, definir tools, mantener estado y aplicar middleware.
- **Proveedor del modelo**, en este caso la integración con Gemini.
- **Sentence Transformers + FAISS**, responsables de la codificación semántica y de la búsqueda vectorial.
- **rank-bm25**, utilizado posteriormente para añadir recuperación léxica.

### Por qué no se instala todo con versiones abiertas

Un notebook que funciona hoy podría dejar de hacerlo si una dependencia introduce un cambio incompatible. Para una entrega evaluable interesa reducir esa fuente de variabilidad. Fijar versiones no garantiza reproducibilidad absoluta —el servicio externo del LLM puede cambiar—, pero sí estabiliza la parte de software que controlamos.

La siguiente celda únicamente instala dependencias; todavía no monta Drive, no descarga modelos y no realiza llamadas a ninguna API.


### Implementación: instalación de dependencias

La celda siguiente utiliza `%pip`, que instala los paquetes en el entorno activo de Colab. Se mantiene separada del resto del setup para que, si el runtime se reinicia, sea evidente qué dependencias deben recuperarse. El mensaje final permite comprobar visualmente que la instalación ha terminado antes de continuar.


In [1]:
%pip install -q \
  langchain==1.3.18 langchain-core==1.6.1 langgraph==1.2.11 \
  langchain-openrouter==0.2.8 langchain-huggingface==1.2.2 \
  sentence-transformers==6.0.1 faiss-cpu==1.15.0 rank-bm25==0.2.2 \
  langchain-google-genai
print("Instalación terminada.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.3/837.3 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 11.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires goo

### Configuración de Drive, rutas y credenciales

A continuación se monta Google Drive y se definen dos rutas con responsabilidades distintas: `DATA_DIR` apunta a la carpeta persistente del proyecto y `CORPUS_DIR` a una carpeta temporal del runtime donde se descomprime el corpus. Esta separación evita trabajar directamente dentro de los ZIP y hace que el acceso a FAISS sea local al entorno de ejecución.

Se añade `DATA_DIR` a `sys.path` para poder importar `miax_s1` y `miax_s2` sin copiar su contenido dentro del notebook. La API key se obtiene mediante `userdata.get`, es decir, desde Secrets de Colab. Si la credencial no existe se produce un error temprano y explícito en lugar de fallar varias celdas después durante una llamada al modelo.

Finalmente se centraliza el nombre del modelo en `MODELO`. De esta forma, cambiar de modelo requiere modificar una única variable y el resto de la arquitectura permanece igual.


In [2]:
from google.colab import drive, userdata
from pathlib import Path
import os, sys, json, time, hashlib, zipfile
import pandas as pd
import numpy as np

drive.mount("/content/drive")

# Carpeta de Drive utilizada durante el desarrollo.
DATA_DIR = Path("/content/drive/MyDrive/MIAX_Taller_NLP")
CORPUS_DIR = Path("/content/corpus")
assert DATA_DIR.exists(), f"No se encuentra la carpeta: {DATA_DIR}"

# Los auxiliares miax_s1.py y miax_s2.py permanecen sin modificar.
if str(DATA_DIR) not in sys.path:
    sys.path.insert(0, str(DATA_DIR))

# Gemini: la clave se guarda en Colab > Secrets como GOOGLE_API_KEY.
try:
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    pass
assert os.environ.get("GOOGLE_API_KEY"), (
    "Falta GOOGLE_API_KEY. Añádela en Colab > Secrets y vuelve a ejecutar."
)

MODELO = "google_genai:gemini-3.8-flash"
print("Entorno configurado. Modelo:", MODELO)


Mounted at /content/drive
Entorno configurado. Modelo: google_genai:gemini-3.8-flash


## 2. Preparación, integridad y carga del corpus

### Objetivo

El agente no trabaja contra Internet ni descarga los 10-K durante la ejecución. Utiliza el corpus proporcionado para la práctica: secciones completas, fragmentos, hechos XBRL y un índice FAISS ya construido. Esta decisión hace que todos los grupos trabajen sobre la misma información y que las métricas sean comparables.

La preparación del corpus se trata como una parte crítica del pipeline. Un índice vectorial solo es correcto si cada vector continúa alineado con el fragmento y los metadatos que le corresponden. Si se mezclasen versiones distintas de `chunks.jsonl` y del índice FAISS, el sistema podría devolver texto incorrecto **sin producir necesariamente un error técnico**. Sería un fallo silencioso y especialmente peligroso en un sistema de recuperación.

### Controles de integridad

Por ello realizamos dos comprobaciones:

1. **SHA-256 de los ZIP originales.** Verifica que los ficheros utilizados son exactamente los esperados y no versiones modificadas o corruptas.
2. **Correspondencia entre chunks e índice.** Tras extraer los ficheros se calcula el hash de `chunks.jsonl` y se contrasta, cuando está disponible, con el manifiesto distribuido junto al índice.

Después se comprueba explícitamente que exista `indice/corpus.faiss`. Solo cuando estas verificaciones pasan consideramos el corpus preparado.

### Estructura lógica de los datos

- `secciones.jsonl`: texto completo de cada sección del 10-K. Alimenta `read_section`.
- `chunks.jsonl`: fragmentos sobre los que opera el retrieval y que poseen `chunk_id` trazable.
- `xbrl_facts.parquet`: hechos financieros estructurados. Es la fuente de verdad para cifras.
- `indice/corpus.faiss`: representación vectorial de los chunks para búsqueda semántica.

Esta separación permite que cada herramienta acceda a la representación más adecuada para su función, en lugar de forzar a todo el sistema a trabajar sobre una única fuente.


### Implementación: extracción segura y comprobación de hashes

La función `sha256()` lee cada fichero por bloques de 1 MB para no cargar el ZIP completo en memoria. Para cada paquete se exige que exista, se calcula su hash y solo si coincide se extrae en `/content/corpus`. Después se verifica la relación entre `chunks.jsonl` y el manifiesto del índice cuando dicho manifiesto está disponible.

El uso de `assert` aquí es intencionado: continuar con datos incorrectos produciría resultados difíciles de diagnosticar, por lo que preferimos detener la ejecución inmediatamente.


In [3]:
PAQUETES = [
    ("corpus_miax_2026.zip", "4233c37fc9e9d12091af7a146063ad70903a3fe51404a485854f4021c63daee4"),
    ("indice_faiss.zip", "6b5610ad8ac6ea50364445d39bb464d993cbd87048fb07c4fe16657d7ac11655"),
]

def sha256(ruta: Path) -> str:
    h = hashlib.sha256()
    with ruta.open("rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()

CORPUS_DIR.mkdir(parents=True, exist_ok=True)
for nombre, esperado in PAQUETES:
    ruta = DATA_DIR / nombre
    assert ruta.is_file(), f"No se encuentra {nombre} en {DATA_DIR}"
    obtenido = sha256(ruta)
    assert obtenido == esperado, f"Hash incorrecto para {nombre}: {obtenido}"
    with zipfile.ZipFile(ruta) as zf:
        zf.extractall(CORPUS_DIR)

hash_chunks = sha256(CORPUS_DIR / "chunks.jsonl")
for manifiesto in ("MANIFEST.md", "indice/MANIFEST.md"):
    ruta = CORPUS_DIR / manifiesto
    if ruta.exists():
        assert hash_chunks in ruta.read_text(encoding="utf-8"), (
            f"El índice no corresponde con chunks.jsonl ({manifiesto})."
        )

assert (CORPUS_DIR / "indice/corpus.faiss").is_file()
print("Corpus e índice preparados en", CORPUS_DIR)


Corpus e índice preparados en /content/corpus


### Carga en memoria y estructuras auxiliares

Con el corpus ya validado se importan los módulos de las dos sesiones y se recargan explícitamente. Esto resulta útil en Colab cuando se ha sustituido un fichero en Drive durante una sesión: `importlib.reload` evita seguir trabajando inadvertidamente con una versión cacheada del módulo.

Después se cargan las tres representaciones principales. `secciones` se convierte en DataFrame porque se consultará por metadatos; `chunks` permanece como lista de diccionarios porque esa estructura encaja con las funciones de retrieval; y `xbrl` se lee desde Parquet, formato adecuado para datos tabulares tipados.

`POR_ID` crea un índice en memoria `chunk_id → fragmento`. Esta estructura permite que el evaluador de citas localice en O(1) el texto al que apunta una respuesta.


In [4]:
import importlib
import miax_s1, miax_s2
importlib.reload(miax_s1)
importlib.reload(miax_s2)

secciones = pd.DataFrame(
    json.loads(linea) for linea in open(CORPUS_DIR / "secciones.jsonl", encoding="utf-8")
)
chunks = [json.loads(linea) for linea in open(CORPUS_DIR / "chunks.jsonl", encoding="utf-8") if linea.strip()]
xbrl = pd.read_parquet(CORPUS_DIR / "xbrl_facts.parquet")
POR_ID = {c["chunk_id"]: c for c in chunks}

print(f"{len(secciones)} secciones · {len(chunks)} chunks · {len(xbrl)} hechos XBRL")
print("Tickers:", sorted(secciones.ticker.unique()))


48 secciones · 1749 chunks · 135 hechos XBRL
Tickers: ['AAPL', 'AMZN', 'GOOGL', 'META', 'MSFT', 'NVDA']


## 3. Cinturón de herramientas: separar fuentes y responsabilidades

### Por qué un agente necesita herramientas especializadas

Un LLM generalista puede redactar respuestas convincentes, pero no debería utilizar su memoria paramétrica para contestar preguntas concretas sobre este corpus. El agente debe apoyarse únicamente en información verificable y disponible en los datos de la práctica. Para ello se exponen cuatro herramientas con responsabilidades diferentes.

Las **firmas de las cuatro herramientas son un contrato externo**: no se cambian sus nombres ni sus parámetros existentes, porque el sistema de evaluación puede invocarlas o inspeccionar su trayectoria. Sí mejoramos su comportamiento y, sobre todo, sus docstrings. Esto último es relevante porque el docstring no es un comentario pasivo: forma parte de la información que recibe el modelo para decidir qué tool debe utilizar.

### 3.1 `list_available()`

Responde a una pregunta de disponibilidad: qué compañías, ejercicios y secciones existen realmente en el corpus. Su función principal es impedir que el agente suponga que dispone de información que no ha sido cargada.

### 3.2 `get_xbrl_fact()`

Es la herramienta exacta para cifras financieras. La decisión metodológica es clara: **una magnitud numérica se obtiene de XBRL y no de la prosa del informe**. Aunque una cifra aparezca también en un fragmento textual, XBRL ofrece una representación estructurada y evita errores causados por tablas partidas, unidades o contexto incompleto.

Además, el concepto XBRL no se infiere por analogía entre compañías. Si un concepto no existe para un ticker y ejercicio concretos, la herramienta devuelve los conceptos disponibles en vez de inventar un sustituto.

### 3.3 `search_filings()`

Se define más adelante, una vez construido el retrieval final. Su misión será resolver preguntas cualitativas mediante recuperación de fragmentos relevantes y devolver siempre `chunk_id` para mantener trazabilidad.

### 3.4 `read_section()`

Permite leer una sección completa. Es deliberadamente una herramienta de último recurso: puede introducir decenas de miles de tokens en el contexto, elevando coste y latencia y dificultando al modelo localizar la evidencia. Por tanto, el prompt indica que se utilice únicamente cuando los fragmentos de `search_filings` sean insuficientes.

La siguiente implementación carga las tres tools que no dependen todavía del retrieval final.


### Implementación de las tools deterministas

La anotación `@tool` convierte cada función Python en una herramienta que LangChain puede exponer al modelo. Es importante observar que el agente no ve el cuerpo de la función: utiliza principalmente nombre, firma y docstring para decidir cuándo llamarla. Por ello los docstrings expresan no solo *qué devuelve* cada función, sino también *cuándo debe utilizarse*.

`get_xbrl_fact` normaliza el ticker a mayúsculas y filtra por las tres claves relevantes: compañía, ejercicio y concepto. Si no encuentra coincidencia, devuelve información explícita sobre conceptos disponibles. Esta respuesta es preferible a lanzar una excepción porque permite al agente corregir una elección de concepto sin romper la trayectoria.

`read_section`, por el contrario, devuelve directamente texto potencialmente largo. La restricción de uso se comunica en el docstring y más adelante también en el system prompt.


In [5]:
from langchain.tools import tool

@tool
def list_available() -> str:
    """Devuelve qué compañías, ejercicios fiscales y secciones existen en el corpus.

    Úsala antes de afirmar que una compañía, ejercicio o sección no está disponible,
    o cuando no estés seguro de qué información puede consultarse.
    """
    lineas = []
    for (ticker, empresa), grupo in secciones.groupby(["ticker", "empresa"]):
        ejercicios = sorted(int(x) for x in grupo["fiscal_year"].unique())
        items = sorted(grupo["item"].unique())
        lineas.append(f"{ticker} ({empresa}): FY{ejercicios}, items {items}")
    return "\n".join(lineas)

@tool
def get_xbrl_fact(ticker: str, fiscal_year: int, concept: str) -> str:
    """Devuelve el valor EXACTO de una magnitud financiera reportada en XBRL.

    Es la fuente autorizada para cualquier cifra. Úsala SIEMPRE para valores
    financieros exactos en lugar de extraer el número de la prosa del 10-K.

    Args:
        ticker: símbolo bursátil, p. ej. 'NVDA'.
        fiscal_year: ejercicio fiscal, p. ej. 2025.
        concept: concepto US-GAAP, p. ej. 'Revenues', 'NetIncomeLoss',
            'Assets' u 'OperatingIncomeLoss'.
    """
    ticker = ticker.upper()
    filas = xbrl[(xbrl.ticker == ticker)
                 & (xbrl.fiscal_year.astype(int) == int(fiscal_year))
                 & (xbrl.concept == concept)]
    if filas.empty:
        disponibles = sorted(xbrl[(xbrl.ticker == ticker)
                                   & (xbrl.fiscal_year.astype(int) == int(fiscal_year))]
                                  .concept.unique())
        if not disponibles:
            return f"No hay datos de {ticker} FY{fiscal_year} en el corpus."
        return (f"{ticker} no reportó '{concept}' en FY{fiscal_year}. "
                f"Conceptos disponibles: {disponibles}")
    f = filas.iloc[0]
    return (f"{ticker} FY{fiscal_year} {concept} = {f['value']:,.0f} {f['unit']} "
            f"(cierre {f['period_end']}, {f['form']})")

@tool
def read_section(ticker: str, fiscal_year: int, item: str) -> str:
    """Devuelve el texto COMPLETO de una sección concreta del 10-K.

    Es una herramienta cara en tokens. Úsala únicamente si search_filings no
    proporciona contexto suficiente y es imprescindible leer la sección completa.
    """
    filas = secciones[(secciones.ticker == ticker.upper())
                      & (secciones.fiscal_year.astype(int) == int(fiscal_year))
                      & (secciones.item == item)]
    if filas.empty:
        return f"No hay Item {item} de {ticker} FY{fiscal_year} en el corpus."
    return filas.iloc[0]["texto"]

print(list_available.invoke({}))


AAPL (Apple Inc.): FY[2024, 2025], items ['1A', '7', '7A', '8']
AMZN (AMAZON COM INC): FY[2024, 2025], items ['1A', '7', '7A', '8']
GOOGL (Alphabet Inc.): FY[2024, 2025], items ['1A', '7', '7A', '8']
META (Meta Platforms, Inc.): FY[2024, 2025], items ['1A', '7', '7A', '8']
MSFT (MICROSOFT CORP): FY[2024, 2025], items ['1A', '7', '7A', '8']
NVDA (NVIDIA CORP): FY[2024, 2025], items ['1A', '7', '7A', '8']


### 3.1 Comparación práctica de las tools: la misma necesidad por el camino correcto y por el incorrecto

Una idea central de la práctica es que **acertar el dato no basta**. El sistema también debe llegar a él por una fuente que generalice. Para una cifra exacta, buscar en prosa es frágil: el número puede aparecer redondeado, en una tabla partida o en un párrafo de otro ejercicio. XBRL, en cambio, es estructurado y determinista. Para una explicación cualitativa ocurre lo contrario: XBRL no contiene el razonamiento de la dirección y necesitamos retrieval.

Las celdas siguientes enseñan las cuatro herramientas con ejemplos y, sobre todo, muestran el fallo que evita cada una.


In [6]:
# EJEMPLO A — cifra exacta: XBRL evita depender de una tabla o párrafo recuperado.
print("A) FUENTE CORRECTA PARA UNA CIFRA")
print(get_xbrl_fact.invoke({"ticker":"NVDA","fiscal_year":2024,"concept":"Revenues"}))

# Enseñamos qué devolvería un retrieval textual para la misma intención.
frag_num = miax_s1.buscar("NVIDIA revenue fiscal 2024", ticker="NVDA", fiscal_year=2024, k=3)
print("\nLa búsqueda textual devuelve contexto, no una garantía de cifra exacta:")
for f in frag_num:
    print(f"- {f['chunk_id']} Item {f['item']}: {f['texto'][:220].replace(chr(10),' ')}")

print("\nProblema evitado: una cifra correcta por casualidad desde la prosa no sustituye una consulta XBRL.")


A) FUENTE CORRECTA PARA UNA CIFRA
NVDA FY2024 Revenues = 60,922,000,000 USD (cierre 2024-01-28, 10-K)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


La búsqueda textual devuelve contexto, no una garantía de cifra exacta:
- NVDA-2024-8-0046 Item 8: Deferred Revenue  The following table shows the changes in deferred revenue during fiscal years 2024 and 2023.  Jan 28, 2024		Jan 29, 2023 (In millions) Balance at beginning of period	$	572			$	502 Deferred revenue addit
- NVDA-2024-8-0066 Item 8: Year Ended Jan 28, 2024		Jan 29, 2023		Jan 30, 2022 Revenue:	(In millions) United States	$	26,966			$	8,292			$	4,349 Taiwan	13,405			6,986			8,544 China (including Hong Kong)	10,306			5,785			7,111 Other countries	10,24
- NVDA-2024-7-0007 Item 7: Market Platform Highlights  Data Center revenue for fiscal year 2024 was $47.5 billion, up 217% from fiscal year 2023. In Data Center, we launched AI inference platforms that combine our full-stack inference software wit

Problema evitado: una cifra correcta por casualidad desde la prosa no sustituye una consulta XBRL.


In [7]:
# EJEMPLO B — dato ausente: get_xbrl_fact debe decir que no existe en vez de estimarlo.
print("B) AUSENCIA EXPLÍCITA EN XBRL")
print(get_xbrl_fact.invoke({"ticker":"AMZN","fiscal_year":2025,"concept":"GrossProfit"}))
print("\nProblema evitado: el agente no debe inferir GrossProfit porque 'parezca calculable' desde otras cifras.")


B) AUSENCIA EXPLÍCITA EN XBRL
AMZN no reportó 'GrossProfit' en FY2025. Conceptos disponibles: ['Assets', 'CashAndCashEquivalentsAtCarryingValue', 'EarningsPerShareBasic', 'EarningsPerShareDiluted', 'NetCashProvidedByUsedInOperatingActivities', 'NetIncomeLoss', 'OperatingIncomeLoss', 'RevenueFromContractWithCustomerExcludingAssessedTax', 'StockholdersEquity']

Problema evitado: el agente no debe inferir GrossProfit porque 'parezca calculable' desde otras cifras.


In [8]:
# EJEMPLO C — disponibilidad: list_available evita responder sobre un documento inexistente.
print("C) CONTROL DE COBERTURA DEL CORPUS")
disponible = list_available.invoke({})
print(disponible)
print("\nProblema evitado: un índice vectorial siempre devuelve algo; comprobar cobertura permite decir 'no está en el corpus'.")


C) CONTROL DE COBERTURA DEL CORPUS
AAPL (Apple Inc.): FY[2024, 2025], items ['1A', '7', '7A', '8']
AMZN (AMAZON COM INC): FY[2024, 2025], items ['1A', '7', '7A', '8']
GOOGL (Alphabet Inc.): FY[2024, 2025], items ['1A', '7', '7A', '8']
META (Meta Platforms, Inc.): FY[2024, 2025], items ['1A', '7', '7A', '8']
MSFT (MICROSOFT CORP): FY[2024, 2025], items ['1A', '7', '7A', '8']
NVDA (NVIDIA CORP): FY[2024, 2025], items ['1A', '7', '7A', '8']

Problema evitado: un índice vectorial siempre devuelve algo; comprobar cobertura permite decir 'no está en el corpus'.


## 4. Retrieval: del baseline denso a un sistema híbrido

### 4.1 Qué queremos mejorar

La búsqueda densa de la primera sesión constituye un baseline razonable: codifica la consulta con BGE, busca por similitud en FAISS y devuelve los fragmentos más próximos. Sin embargo, una búsqueda semántica pura tiene limitaciones previsibles en este corpus.

Los 10-K contienen lenguaje repetitivo, especialmente entre ejercicios de una misma compañía. Además, algunas consultas dependen de señales muy literales —nombres propios, años, porcentajes, términos financieros o expresiones exactas— que BM25 puede capturar mejor que un embedding. Finalmente, si ya conocemos ticker, ejercicio o sección, hacer competir el fragmento correcto contra los 1.749 chunks introduce ruido innecesario.

Por ello se estudian tres mejoras acumulativas:

1. **Filtrado por metadatos.** Restringir los candidatos por `ticker`, `fiscal_year` e `item` cuando esa información está disponible.
2. **Retrieval híbrido.** Combinar el ranking semántico de FAISS con un ranking léxico BM25.
3. **Reescritura de la consulta.** Traducir y adaptar la pregunta al vocabulario probable del 10-K antes de recuperar.

### 4.2 Por qué conservar el baseline

No sustituimos directamente la búsqueda inicial. La mantenemos como función separada porque la práctica exige demostrar si las mejoras **mueven realmente la métrica**. Sin baseline no existiría un punto de comparación y no podríamos distinguir una mejora real de una arquitectura simplemente más compleja.

### 4.3 Filtro por metadatos

El filtro no pretende mejorar el embedding; modifica el **espacio de candidatos**. Si la pregunta especifica Microsoft FY2025, un fragmento casi idéntico de FY2024 no debería competir en igualdad de condiciones. El índice sigue calculando el ranking, pero solo se aceptan posiciones cuyos metadatos cumplen las restricciones conocidas.

### 4.4 BM25 + denso mediante Reciprocal Rank Fusion

FAISS devuelve similitud semántica y BM25 devuelve una puntuación léxica. Sus escalas no son directamente comparables, por lo que sumarlas sería arbitrario. Utilizamos **Reciprocal Rank Fusion (RRF)**, que combina la posición relativa de cada documento en ambos rankings:

\[ RRF(d)=\sum_r \frac{1}{k + rank_r(d)} \]

Así, un fragmento bien posicionado por ambos métodos recibe más peso, pero un resultado especialmente fuerte en uno de ellos también puede aparecer entre los primeros. El hiperparámetro `rrf_k=60` suaviza el efecto de las primeras posiciones y es un valor habitual para esta técnica.

La implementación mantiene los metadatos y `chunk_id` de cada fragmento para que la mejora de retrieval no rompa la trazabilidad posterior.


### Implementación de los retrievers baseline, filtrado e híbrido

La siguiente celda define tres funciones separadas para poder medir cada estrategia. `buscar_denso_baseline` delega en `miax_s1.buscar`, preservando el comportamiento original. `buscar_denso_filtrado` construye una máscara booleana sobre los metadatos y mantiene únicamente candidatos compatibles.

`buscar_hibrido` ejecuta dos rankings independientes. Para la rama densa utiliza el índice FAISS y para la rama léxica obtiene scores BM25 sobre los chunks originales. Ambos rankings se restringen al mismo universo permitido por metadatos. Finalmente RRF fusiona **rangos**, no scores, evitando mezclar magnitudes que no tienen la misma interpretación estadística.

El resultado conserva el formato de fragmento esperado por el resto del sistema, de modo que cambiar el retriever no obliga a reescribir tools o evaluadores. Esta compatibilidad de interfaces es una decisión importante para mantener modularidad.


In [9]:
# Baseline denso original: se conserva exactamente como referencia.
def buscar_denso_baseline(query: str, ticker=None, fiscal_year=None, item=None, k: int = 5):
    return miax_s1.buscar(query, ticker=ticker, fiscal_year=fiscal_year, item=item, k=k)

# Arreglo 1: búsqueda densa + filtro explícito de metadatos.
def buscar_denso_filtrado(query: str, ticker=None, fiscal_year=None, item=None, k: int = 5):
    indice, meta, _ = miax_s2.cargar_indice()
    permitidas = np.ones(len(meta), dtype=bool)
    if ticker is not None:
        permitidas &= meta["ticker"].eq(ticker.upper()).to_numpy()
    if fiscal_year is not None:
        permitidas &= meta["fiscal_year"].astype(int).eq(int(fiscal_year)).to_numpy()
    if item is not None:
        permitidas &= meta["item"].eq(item).to_numpy()

    scores, posiciones = indice.search(miax_s2.codificar([query]), indice.ntotal)
    salida = []
    for score, pos in zip(scores[0], posiciones[0]):
        pos = int(pos)
        if pos < 0 or not permitidas[pos]:
            continue
        salida.append(miax_s2.fila_a_fragmento(meta.iloc[pos], score))
        if len(salida) >= k:
            break
    return salida

# Arreglo 2: BM25 + denso mediante Reciprocal Rank Fusion.
def buscar_hibrido(query: str, ticker=None, fiscal_year=None, item=None,
                   k: int = 5, rrf_k: int = 60):
    indice, meta, _ = miax_s2.cargar_indice()

    # Universo permitido por metadatos.
    permitidas = np.ones(len(meta), dtype=bool)
    if ticker is not None:
        permitidas &= meta["ticker"].eq(ticker.upper()).to_numpy()
    if fiscal_year is not None:
        permitidas &= meta["fiscal_year"].astype(int).eq(int(fiscal_year)).to_numpy()
    if item is not None:
        permitidas &= meta["item"].eq(item).to_numpy()
    ids_permitidos = set(meta.loc[permitidas, "chunk_id"])

    # Ranking denso completo.
    scores_d, pos_d = indice.search(miax_s2.codificar([query]), indice.ntotal)
    rank_d = {}
    frag_por_id = {}
    for rank, (score, pos) in enumerate(zip(scores_d[0], pos_d[0]), 1):
        pos = int(pos)
        if pos < 0:
            continue
        cid = meta.iloc[pos]["chunk_id"]
        if cid in ids_permitidos:
            rank_d[cid] = rank
            frag_por_id[cid] = miax_s2.fila_a_fragmento(meta.iloc[pos], score)

    # Ranking BM25 sobre los chunks originales.
    bm25, chunks_bm = miax_s2.montar_bm25()
    scores_bm = bm25.get_scores(miax_s2.tokenizar(query))
    orden_bm = np.argsort(scores_bm)[::-1]
    rank_b = {}
    for rank, pos in enumerate(orden_bm, 1):
        c = chunks_bm[int(pos)]
        cid = c["chunk_id"]
        if cid in ids_permitidos:
            rank_b[cid] = rank
            if cid not in frag_por_id:
                frag_por_id[cid] = {**c, "puntuacion": float(scores_bm[int(pos)])}

    # RRF combina posiciones, no escalas incompatibles de score.
    candidatos = ids_permitidos & (set(rank_d) | set(rank_b))
    fusion = {
        cid: 1/(rrf_k + rank_d.get(cid, 10**6)) + 1/(rrf_k + rank_b.get(cid, 10**6))
        for cid in candidatos
    }
    mejores = sorted(fusion, key=fusion.get, reverse=True)[:k]
    salida = []
    for cid in mejores:
        f = dict(frag_por_id[cid])
        f["puntuacion"] = round(float(fusion[cid]), 6)
        salida.append(f)
    return salida

print("Retrievers definidos.")


Retrievers definidos.


### Reescritura de consultas y definición de `search_filings`

La pregunta del usuario puede estar formulada en español y con vocabulario conversacional, mientras que los documentos 10-K están en inglés y utilizan terminología financiera específica. El reescritor recibe una instrucción muy restringida: producir únicamente una consulta de búsqueda en inglés conservando compañía, ejercicio y concepto. No se le pide responder a la pregunta; su función es mejorar la representación de la necesidad de información.

La llamada está protegida con `try/except`. Si el proveedor falla, el sistema degrada de forma controlada y utiliza la pregunta original. Así, una mejora opcional de retrieval no se convierte en un punto único de fallo de todo el agente.

`search_filings` será la tool cualitativa del sistema final: reescribe y posteriormente recupera mediante el retriever híbrido, manteniendo los filtros que el agente haya inferido de la pregunta.


In [10]:
from langchain.chat_models import init_chat_model

INSTRUCCION_REESCRITURA = """Reescribe la pregunta como una consulta de búsqueda
para un índice de informes 10-K en INGLÉS. Conserva compañía, ejercicio y concepto,
y usa vocabulario probable del propio informe. Devuelve SOLO la consulta."""

reescritor = init_chat_model(MODELO, temperature=0)

def reescribir_consulta(pregunta: str) -> str:
    try:
        return reescritor.invoke([
            {"role": "system", "content": INSTRUCCION_REESCRITURA},
            {"role": "user", "content": pregunta},
        ]).text.strip()
    except Exception:
        # El agente final también recibe la instrucción de buscar en inglés.
        return pregunta

@tool
def search_filings(query: str, ticker: str | None = None,
                   fiscal_year: int | None = None,
                   item: str | None = None, k: int = 5) -> str:
    """Busca fragmentos relevantes en los 10-K mediante retrieval híbrido.

    Úsala para riesgos, estrategia, litigios, explicaciones del MD&A y demás
    preguntas cualitativas. NO la uses como fuente autorizada de cifras: para
    valores exactos usa get_xbrl_fact.

    Args:
        query: consulta de búsqueda, preferentemente en inglés y con vocabulario del 10-K.
        ticker: filtra por compañía si se conoce.
        fiscal_year: filtra por ejercicio si se conoce.
        item: filtra por sección ('1A', '7', '7A' u '8') si se conoce.
        k: número de fragmentos a devolver.
    """
    fragmentos = buscar_hibrido(query, ticker, fiscal_year, item, k)
    return miax_s2.formatear_fragmentos(fragmentos)

HERRAMIENTAS = [list_available, get_xbrl_fact, search_filings, read_section]
print("Herramientas finales:", [t.name for t in HERRAMIENTAS])


Herramientas finales: ['list_available', 'get_xbrl_fact', 'search_filings', 'read_section']


### 4.6 Golden set propio: conjunto de evaluación de la entrega

A partir de este punto utilizamos **un único golden set externo**: las 20 preguntas construidas por el grupo para esta práctica y almacenadas exclusivamente en `golden_set.jsonl`. No introducimos un segundo conjunto “oficial” ni mezclamos preguntas de clase con la evaluación de la entrega.

El golden set cumple tres funciones. Primero, fija de antemano qué consideramos una respuesta correcta. Segundo, permite comparar el baseline y las sucesivas mejoras sobre exactamente los mismos casos. Tercero, evita evaluar el sistema únicamente con ejemplos escogidos después de ver sus respuestas.

El enunciado exige 20 preguntas y al menos 6 comparativas. Organizamos el conjunto en tres familias: **numéricas**, que deben resolverse contra XBRL; **extractivas**, que requieren recuperar evidencia textual del 10-K; y **comparativas**, que obligan a combinar información de más de un ejercicio y, en varios casos, evidencia numérica y textual.

Aparte del golden utilizaremos algunos **casos diagnósticos**. Estos no cuentan para la métrica final: sirven únicamente para visualizar un fallo concreto —por ejemplo, recuperar la empresa equivocada o leer una cifra desde prosa— y comprobar qué componente lo corrige.


### 4.7 Golden set como artefacto externo, no hardcodeado

Las 20 preguntas de evaluación viven **únicamente en `golden_set.jsonl`**. El notebook no mantiene una segunda copia en una lista Python: duplicar el conjunto en dos sitios haría muy fácil modificar una versión y evaluar accidentalmente con otra. Por eso esta sección se limita a cargar el JSONL, enriquecer en memoria los campos que pueden contrastarse automáticamente contra XBRL y validarlo.

El conjunto se ha escrito además con formulaciones deliberadamente heterogéneas. No todas las preguntas dicen `FY2025`: aparecen expresiones como *durante 2025*, *entre 2024 y 2025* o *en el informe correspondiente a 2025*. Esto evita adaptar el agente a una plantilla lingüística artificial. Algunas preguntas numéricas exigen dos consultas XBRL y calcular una variación entre ejercicios. También incluimos casos fuera del corpus —un ticker inexistente y un ejercicio no disponible— para comprobar que el agente detecta los límites del corpus y **no fabrica una cifra**.

Para comparativas numéricas guardamos `fiscal_years_esperados`, mientras que `fiscal_year` conserva el ejercicio principal para mantener compatibilidad con el resto de la infraestructura. Los valores esperados se obtienen del parquet XBRL al ejecutar el notebook; no se escriben a mano.


In [11]:
RUTA_GOLDEN = DATA_DIR / "golden_set.jsonl"

if not RUTA_GOLDEN.is_file():
    raise FileNotFoundError(
        f"No se encuentra {RUTA_GOLDEN}. Copia golden_set.jsonl a la carpeta MIAX_Taller_NLP de Drive. "
        "El golden set no está hardcodeado en este notebook."
    )

golden = [json.loads(l) for l in RUTA_GOLDEN.open(encoding="utf-8") if l.strip()]

def _fila_xbrl(ticker, fy, concept):
    f = xbrl[(xbrl.ticker == ticker) &
             (xbrl.fiscal_year.astype(int) == int(fy)) &
             (xbrl.concept == concept)]
    return None if f.empty else f.iloc[0]

def enriquecer_ground_truth_xbrl(preguntas):
    """Añade en memoria el ground truth numérico directamente desde XBRL.

    - Una pregunta numérica simple recibe cifra_esperada/unidad.
    - Una comparativa de años recibe cifras_esperadas={año: valor} y una respuesta
      de referencia con diferencia absoluta y porcentual.
    - Si espera ausencia de datos, NO intenta fabricar un valor.
    El JSONL original no se reescribe: sigue siendo la única definición de preguntas.
    """
    salida=[]
    for p0 in preguntas:
        p=dict(p0)
        if p.get("espera_sin_datos"):
            p["respuesta_esperada"] = "Dato no disponible en el corpus"
            salida.append(p); continue
        concept=p.get("concept_xbrl")
        years=p.get("fiscal_years_esperados") or ([p.get("fiscal_year")] if p.get("fiscal_year") else [])
        if concept and years:
            vals={}
            units=[]
            for fy in years:
                f=_fila_xbrl(p["ticker"], fy, concept)
                if f is not None:
                    vals[int(fy)] = float(f["value"]); units.append(str(f["unit"]))
            if len(years)>1 and len(vals)==len(years):
                p["cifras_esperadas"] = vals
                a,b=map(int,years[:2]); va,vb=vals[a],vals[b]
                delta=vb-va; pct=(delta/va*100) if va else None
                p["unidad"] = units[0] if units else None
                p["respuesta_esperada"] = (
                    f"{a}: {va:,.0f}; {b}: {vb:,.0f}; variación: {delta:,.0f} "
                    + (f"({pct:.2f}%)" if pct is not None else "")
                )
            elif len(vals)==1:
                fy=next(iter(vals)); p["cifra_esperada"]=vals[fy]
                p["unidad"] = units[0] if units else None
                p["respuesta_esperada"] = f"{vals[fy]:,.0f} {p['unidad']}"
        salida.append(p)
    return salida

golden = enriquecer_ground_truth_xbrl(golden)

print(f"Golden cargado exclusivamente desde: {RUTA_GOLDEN}")
print(pd.DataFrame(golden)[["id","familia","pregunta","ticker","fiscal_year","fiscal_years_esperados","espera_sin_datos"]].to_string(index=False))
print("\nDistribución:", pd.Series([p["familia"] for p in golden]).value_counts().to_dict())


Golden cargado exclusivamente desde: /content/drive/MyDrive/MIAX_Taller_NLP/golden_set.jsonl
    id     familia                                                                                                                           pregunta ticker  fiscal_year fiscal_years_esperados  espera_sin_datos
g3-001    numerica                                     ¿Cuánto aumentaron los ingresos de NVIDIA de 2024 a 2025, tanto en dólares como en porcentaje?   NVDA         2025           [2024, 2025]             False
g3-002    numerica                                                                           Durante 2025, ¿cuál fue el revenue reportado por NVIDIA?   NVDA         2025                   None             False
g3-003    numerica                     ¿Cómo cambió el beneficio neto de Microsoft entre 2024 y 2025? Indica ambos valores y la variación porcentual.   MSFT         2025           [2024, 2025]             False
g3-004    numerica                                             

**Control manual imprescindible para las preguntas textuales.** El código anterior puede contrastar automáticamente las cifras porque XBRL es estructurado. No hacemos lo mismo con las respuestas extractivas: escoger como “verdad” el primer fragmento que devuelve nuestro propio retriever produciría una evaluación circular. Por tanto, las siete extractivas y la parte textual de las comparativas deben revisarse contra el 10-K y completar `ancla_texto` con **una frase literal**. El notebook muestra a continuación candidatos, pero no los convierte silenciosamente en ground truth.


In [12]:
# Asistente de revisión: enseña candidatos, pero NO modifica el golden.
# Sirve para que el grupo copie una frase literal tras comprobarla visualmente.
for p in [x for x in golden if x["familia"] in {"extractiva","comparativa"}]:
    q = miax_s2.REESCRITURAS_RESPALDO.get(p["id"], p["pregunta"])
    candidatos = buscar_denso_filtrado(q, p["ticker"], p["fiscal_year"], p.get("item_esperado"), 3)
    print("\n" + "="*100)
    print(p["id"], p["pregunta"])
    for c in candidatos:
        print(f"\n[{c['chunk_id']}] {c['ticker']} FY{c['fiscal_year']} Item {c['item']}")
        print(c["texto"][:900].replace("\n"," "))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


g3-008 En el informe correspondiente a 2025, ¿qué riesgos señala NVIDIA sobre las restricciones de exportación y la competencia en China?

[NVDA-2025-1A-0034] NVDA FY2025 Item 1A
, and D5, excluding Israel. To date, we have not received licenses to ship these restricted products to China.  On January 15, 2025, the USG published the “AI Diffusion” IFR in the Federal Register. After a 120-day delayed compliance period, the IFR will, unless modified, impose a worldwide licensing requirement on all products classified under Export Control Classification Numbers, or ECCNs, 3A090.a, 4A090.a, or corresponding .z ECCNs, including all related software and technology. Any system that incorporates one or more of the covered integrated circuits, or ICs, (including but not limited to NVIDIA DGX, HGX, and MGX systems) will be covered by the new licensing requirement. The licensing requirement will include future NVIDIA ICs, boards, or systems classified with ECCN 3A090.a or 4A090.a, or correspondin

### 4.8 Validación fuerte del JSON antes de medir

El validador diferencia entre **estructura válida** y **golden realmente cerrado**. La primera comprobación exige 20 preguntas, IDs únicos y al menos seis comparativas. La segunda, que activaremos para la entrega definitiva, exige además que toda extractiva tenga `ancla_texto` literal y que toda pregunta numérica tenga una cifra contrastada. De esta forma el notebook puede utilizarse mientras el grupo revisa las anclas sin fingir que el trabajo de etiquetado ya está terminado.


In [13]:
CAMPOS_GOLDEN = {"id","pregunta","familia","ticker","fiscal_year","fiscal_years_esperados",
                 "respuesta_esperada","cifra_esperada","cifras_esperadas","unidad","concept_xbrl",
                 "item_esperado","ancla_texto","ancla_inicio","ancla_fin","chunk_id_esperado",
                 "herramienta_esperada","espera_sin_datos","autor"}

def validar_golden(preguntas, exigir_anclas=False):
    problemas=[]
    if len(preguntas)!=20: problemas.append(f"Se requieren 20 preguntas; hay {len(preguntas)}")
    if sum(p.get("familia")=="comparativa" for p in preguntas)<6:
        problemas.append("Faltan comparativas: mínimo 6")
    ids=[p.get("id") for p in preguntas]
    if len(ids)!=len(set(ids)): problemas.append("Hay IDs repetidos")
    for p in preguntas:
        faltan=CAMPOS_GOLDEN-set(p)
        if faltan: problemas.append(f"{p.get('id')}: faltan {sorted(faltan)}")
        if p.get("espera_sin_datos"):
            if p.get("cifra_esperada") is not None or p.get("cifras_esperadas"):
                problemas.append(f"{p['id']}: un caso sin datos no debe tener cifra esperada")
            continue
        years=p.get("fiscal_years_esperados") or []
        if p.get("concept_xbrl") and len(years)>1 and not p.get("cifras_esperadas"):
            problemas.append(f"{p['id']}: comparativa XBRL sin las cifras de ambos ejercicios")
        elif p.get("concept_xbrl") and len(years)<=1 and p.get("cifra_esperada") is None:
            problemas.append(f"{p['id']}: concepto XBRL sin cifra contrastada")
        if exigir_anclas and p.get("familia") in {"extractiva","comparativa"} and not p.get("ancla_texto"):
            problemas.append(f"{p['id']}: falta ancla_texto revisada")
    return problemas

problemas = validar_golden(golden, exigir_anclas=False)
print("Estructura del golden:", "OK" if not problemas else "\n- " + "\n- ".join(problemas))
pendientes=[p["id"] for p in golden if p["familia"] in {"extractiva","comparativa"} and not p.get("ancla_texto")]
print("Anclas textuales pendientes de revisión manual:", pendientes or "ninguna")


Estructura del golden: OK
Anclas textuales pendientes de revisión manual: ['g3-008', 'g3-009', 'g3-010', 'g3-011', 'g3-012', 'g3-013', 'g3-014', 'g3-015', 'g3-016', 'g3-017', 'g3-018', 'g3-019', 'g3-020']


### 4.9 Ablation study del retrieval: ver qué aporta cada decisión

No basta con comparar “baseline” y “final”. Si cambiamos a la vez filtros, BM25 y reescritura, una mejora final no nos dice **qué componente fue responsable**. Por eso hacemos una ablación incremental:

1. **Denso plano**: lo que teníamos al principio.
2. **+ metadatos**: evita mezclar compañías, ejercicios y secciones.
3. **+ BM25/RRF**: añade coincidencia léxica exacta.
4. **+ reescritura**: traduce la pregunta al idioma y vocabulario del 10-K.
5. **Reescritura + híbrido**: configuración final de retrieval.

Para cada pregunta con ancla medimos `hit@5` y, además, la **posición exacta del ancla**. Esto permite distinguir un fallo leve (ancla en posición 6) de uno estructural (posición 700).


In [14]:
def ranking_completo(buscar_fn, p, query):
    # Pedimos suficientes candidatos para poder observar la posición del ancla.
    try:
        return buscar_fn(query, p.get("ticker"), p.get("fiscal_year"), p.get("item_esperado"), 2000)
    except TypeError:
        return buscar_fn(query, p.get("ticker"), p.get("fiscal_year"), p.get("item_esperado"), k=2000)

def comparar_retrieval(preguntas):
    evaluables=[p for p in preguntas if p.get("ancla_texto")]
    if not evaluables:
        print("Aún no hay anclas textuales cerradas: la tabla se activará al completarlas.")
        return pd.DataFrame()
    configs={
        "1_denso_plano": lambda p: buscar_denso_baseline(p["pregunta"], k=2000),
        "2_metadatos": lambda p: buscar_denso_filtrado(p["pregunta"],p["ticker"],p["fiscal_year"],p.get("item_esperado"),2000),
        "3_hibrido": lambda p: buscar_hibrido(p["pregunta"],p["ticker"],p["fiscal_year"],p.get("item_esperado"),2000),
        "4_rewrite_denso": lambda p: buscar_denso_filtrado(reescribir_consulta(p["pregunta"]),p["ticker"],p["fiscal_year"],p.get("item_esperado"),2000),
        "5_rewrite_hibrido": lambda p: buscar_hibrido(reescribir_consulta(p["pregunta"]),p["ticker"],p["fiscal_year"],p.get("item_esperado"),2000),
    }
    filas=[]
    for p in evaluables:
        fila={"id":p["id"],"ticker":p["ticker"],"item":p.get("item_esperado")}
        for nombre,fn in configs.items():
            t0=time.perf_counter(); rec=fn(p); dt=time.perf_counter()-t0
            pos=miax_s2.posicion_del_ancla(p,rec)
            fila[nombre+"_hit@5"] = pos is not None and pos<=5
            fila[nombre+"_rank"] = pos
            fila[nombre+"_ms"] = 1000*dt
        filas.append(fila)
    return pd.DataFrame(filas)

tabla_ablation = comparar_retrieval(golden)
if not tabla_ablation.empty:
    display(tabla_ablation)
    resumen=[]
    for pref in ["1_denso_plano","2_metadatos","3_hibrido","4_rewrite_denso","5_rewrite_hibrido"]:
        resumen.append({"configuración":pref,
                        "recall@5":tabla_ablation[pref+"_hit@5"].mean(),
                        "latencia retrieval ms":tabla_ablation[pref+"_ms"].mean()})
    display(pd.DataFrame(resumen).style.format({"recall@5":"{:.1%}","latencia retrieval ms":"{:.1f}"}))


Aún no hay anclas textuales cerradas: la tabla se activará al completarlas.


### 4.10 Ejemplo visual: el mismo caso antes y después de cada mejora

La tabla agregada es necesaria, pero para defender la arquitectura conviene poder **ver** el fallo. La siguiente función imprime los cinco primeros resultados de una misma pregunta bajo varias configuraciones. Así se observa, por ejemplo, si el baseline devuelve otro ejercicio, si el filtro elimina ese ruido o si la reescritura en inglés hace aparecer el párrafo correcto.

Este ejemplo es especialmente útil en la presentación: en lugar de afirmar “los metadatos mejoran el retrieval”, enseñamos qué documento incorrecto desaparece del top-5 y dónde aparece el ancla correcta.


In [15]:
def demo_retrieval(p, k=5):
    variantes=[
        ("DENSO PLANO", p["pregunta"], lambda q: buscar_denso_baseline(q,k=k)),
        ("+ METADATOS", p["pregunta"], lambda q: buscar_denso_filtrado(q,p["ticker"],p["fiscal_year"],p.get("item_esperado"),k)),
        ("+ HÍBRIDO", p["pregunta"], lambda q: buscar_hibrido(q,p["ticker"],p["fiscal_year"],p.get("item_esperado"),k)),
        ("+ REWRITE + HÍBRIDO", reescribir_consulta(p["pregunta"]), lambda q: buscar_hibrido(q,p["ticker"],p["fiscal_year"],p.get("item_esperado"),k)),
    ]
    print("PREGUNTA:",p["pregunta"])
    if p.get("ancla_texto"): print("ANCLA:",p["ancla_texto"][:220])
    for nombre,q,fn in variantes:
        print("\n"+"="*90+f"\n{nombre}\nquery -> {q}")
        rec=fn(q)
        for j,f in enumerate(rec,1):
            hit = "  <-- CONTIENE ANCLA" if p.get("ancla_texto") and miax_s2.acierta(p,[f]) else ""
            print(f"{j}. {f['chunk_id']} | {f['ticker']} FY{f['fiscal_year']} Item {f['item']} | {f['puntuacion']:.4f}{hit}")
            print("   "+f["texto"][:180].replace("\n"," "))

# Elegimos automáticamente la primera pregunta extractiva cerrada.
ejemplo = next((p for p in golden if p.get("ancla_texto")), None)
if ejemplo:
    demo_retrieval(ejemplo)
else:
    print("Completa al menos un ancla_texto para activar la demo visual.")


Completa al menos un ancla_texto para activar la demo visual.


## 4.11 Qué problemas hemos ido eliminando: resumen de la capa de retrieval

Antes de construir el agente conviene detenerse y leer la secuencia causal. Cada componente existe porque observamos un fallo concreto:

| Cambio | Fallo que ataca | Evidencia que buscamos |
|---|---|---|
| Filtro por metadatos | recupera compañía/año/sección equivocada | desaparecen documentos incompatibles del top-k |
| BM25 + RRF | embeddings pierden términos literales | sube el ranking de coincidencias exactas |
| Reescritura a inglés | pregunta española vs corpus/embedding inglés | el ancla entra en top-5 o mejora su posición |
| Agente | una comparativa necesita varios pasos | trayectoria con varias tools/ejercicios |
| XBRL guardrail | cifra plausible pero no verificada | detección del desajuste y nueva iteración |
| Límites | bucle ReAct sin convergencia | máximo acotado de llamadas |

La utilidad de esta tabla es metodológica: **no añadimos componentes porque “suenen avanzados”**, sino porque existe un error observable que cada uno intenta reducir. Y si una mejora no mueve la métrica —como puede ocurrir con BM25 antes de resolver el idioma— se conserva el resultado negativo como parte de la evaluación.


## 5. Construcción del agente final y contrato de salida

### Del retrieval a un agente

Hasta este punto hemos construido fuentes de información y mecanismos de recuperación. El siguiente nivel consiste en permitir que un modelo decida **qué fuente necesita y en qué orden utilizarla**. Esa es la función del agente.

El enrutamiento esperado es deliberadamente asimétrico:

- para una **cifra exacta**, `get_xbrl_fact`;
- para una **pregunta cualitativa**, `search_filings`;
- para una sección completa, `read_section`, únicamente si la búsqueda por fragmentos no basta;
- para comprobar disponibilidad, `list_available`.

Las preguntas comparativas añaden una dificultad adicional: una respuesta sobre el cambio entre FY2024 y FY2025 no debe basarse en una única recuperación. El agente debe descomponer el problema, consultar ambos ejercicios y después sintetizar la comparación.

### Salida estructurada como mecanismo de ingeniería, no solo de presentación

La respuesta final se modela con Pydantic. Esto obliga al agente a devolver campos estables: `respuesta`, `cifra`, `unidad`, `ticker`, `ejercicio`, `fuente`, `cita` y `chunk_id`. La estructura tiene tres ventajas:

1. **Trazabilidad:** sabemos qué fuente declara haber utilizado y qué fragmento cita.
2. **Evaluación automática:** los evaluadores no tienen que extraer información de prosa arbitraria.
3. **Guardrails:** podemos inspeccionar programáticamente una cifra antes de aceptar la respuesta.

El texto de `respuesta` sigue siendo legible para el usuario, pero deja de ser el único artefacto del sistema.


### Definición del esquema `RespuestaFinanciera` y del system prompt

Pydantic actúa como contrato entre el agente y los evaluadores. Los campos opcionales permiten representar correctamente preguntas cualitativas, para las que puede no existir una cifra, y ausencias reales de datos, para las que la fuente puede ser `ninguna`.

El system prompt contiene reglas de routing y de trazabilidad. Se insiste en utilizar XBRL para cifras, retrieval para texto y consultas en inglés para el corpus. Estas instrucciones no sustituyen a los guardrails posteriores; sirven como primera capa de control, mientras que el middleware implementa comprobaciones que no dependen únicamente de obediencia del modelo.


In [16]:
from typing import Literal
from pydantic import BaseModel, Field

class RespuestaFinanciera(BaseModel):
    """Respuesta trazable a una pregunta sobre informes 10-K."""
    respuesta: str = Field(description="Respuesta breve, directa y sustentada en las tools")
    cifra: float | None = Field(default=None, description="Valor numérico principal si aplica")
    unidad: str | None = Field(default=None, description="USD, shares, porcentaje, etc.")
    ticker: str | None = None
    ejercicio: int | None = None
    fuente: Literal["xbrl", "texto", "ambas", "ninguna"]
    cita: str | None = Field(default=None, description="Texto literal que respalda la respuesta")
    chunk_id: str | None = Field(default=None, description="Chunk citado cuando la fuente incluye texto")

SYSTEM = """Eres un analista financiero que responde exclusivamente con la información
obtenida mediante las herramientas disponibles sobre informes 10-K.

Reglas obligatorias:
1. Para CUALQUIER cifra financiera exacta usa get_xbrl_fact. No tomes cifras de la prosa.
2. Para riesgos, estrategia, litigios, causas o comentarios de dirección usa search_filings.
3. Formula las consultas de search_filings en INGLÉS, aunque el usuario pregunte en español.
4. Pasa ticker, fiscal_year e item como filtros siempre que puedan deducirse de la pregunta.
5. Para una comparativa entre FY2024 y FY2025 consulta los dos ejercicios; si además pide el porqué,
   combina XBRL para las cifras y search_filings para la explicación.
6. Usa read_section solo si los fragmentos recuperados no bastan.
7. Si no sabes si el dato existe, usa list_available. Si no está en el corpus, dilo y no lo estimes.
8. Cuando uses texto, devuelve chunk_id y una cita breve literalmente respaldada por ese chunk.
9. No inventes conceptos XBRL. Si un concepto no existe, revisa los conceptos que devuelve la tool.
"""
print("Contrato de respuesta definido.")


Contrato de respuesta definido.


## 6. Guardrails y middleware: limitar y verificar el comportamiento

### 6.1 Por qué hacen falta guardrails

Un agente puede entrar en bucles, repetir herramientas, realizar demasiadas llamadas o producir una cifra que no coincide con la fuente consultada. Estos fallos no se resuelven únicamente con un mejor prompt: conviene imponer controles programáticos alrededor del modelo.

Se incorporan dos tipos de protección.

### Límites operativos

Se limita el número de llamadas al modelo y a herramientas por invocación. El objetivo es doble: evitar trayectorias degeneradas y mantener controlados **coste y latencia**. Una respuesta que termina siendo correcta después de una cantidad desproporcionada de llamadas no es equivalente a una solución eficiente.

### Verificación de cifras contra XBRL

El middleware propio inspecciona la salida estructurada. Si el modelo afirma una cifra, se contrasta contra los hechos XBRL correspondientes al ticker y ejercicio. La comparación admite una tolerancia relativa pequeña para no penalizar redondeos razonables.

Si la cifra no coincide con ningún hecho reportado, el sistema no acepta inmediatamente la respuesta. Devuelve información del desajuste al modelo y le concede una oportunidad de corregirla. Se evita repetir indefinidamente esta corrección: el guardrail debe aumentar robustez, no crear otro bucle.

### Qué problema evita

Este diseño separa **generación** de **validación**. El LLM puede razonar y redactar, pero no tiene autoridad final sobre una cifra financiera cuando existe una fuente estructurada contra la que contrastarla. Es una aplicación concreta del principio de usar determinismo allí donde el problema lo permite.


### Implementación del middleware y límites

Los middlewares de límite controlan el presupuesto máximo de interacción. El middleware propio, ejecutado después de la generación, inspecciona la salida estructurada y consulta los hechos XBRL cargados localmente. La función `cuadra` aplica tolerancia relativa, lo que permite diferencias de redondeo sin aceptar desviaciones sustanciales.

La lógica de corrección debe ser conservadora: solo interviene cuando existe una cifra verificable y suficiente contexto de ticker/ejercicio. En preguntas sin cifra no intenta forzar una validación numérica.


In [17]:
from langchain.agents.middleware import (
    AgentState, after_model, ToolCallLimitMiddleware, ModelCallLimitMiddleware
)
from langgraph.runtime import Runtime

TOLERANCIA = 0.01
MARCA = "VERIFICACIÓN AUTOMÁTICA XBRL"

@after_model(can_jump_to=["model"])
def verificar_cifras_contra_xbrl(state: AgentState, runtime: Runtime) -> dict | None:
    respuesta = state.get("structured_response")
    if respuesta is None or getattr(respuesta, "cifra", None) is None:
        return None
    ticker = getattr(respuesta, "ticker", None)
    ejercicio = getattr(respuesta, "ejercicio", None)
    if not ticker or ejercicio is None:
        return None

    # Una sola corrección automática por invocación.
    for m in state.get("messages", []):
        contenido = str(getattr(m, "content", ""))
        if MARCA in contenido:
            return None

    hechos = xbrl[(xbrl.ticker == str(ticker).upper())
                  & (xbrl.fiscal_year.astype(int) == int(ejercicio))]
    if hechos.empty:
        return None

    afirmada = float(respuesta.cifra)
    if any(miax_s2.cuadra(afirmada, float(v), TOLERANCIA) for v in hechos["value"]):
        return None

    disponibles = ", ".join(
        f"{r.concept}={float(r.value):,.0f} {r.unit}"
        for _, r in hechos.iterrows()
    )
    mensaje = (
        f"{MARCA}: afirmaste {afirmada:,.6g} para {ticker} FY{ejercicio}, "
        "pero esa cifra no coincide (tolerancia 1%) con ningún hecho XBRL reportado. "
        f"Hechos disponibles: {disponibles}. Corrige la respuesta usando get_xbrl_fact "
        "o indica que el dato solicitado no está disponible."
    )
    return {"messages": [{"role": "user", "content": mensaje}], "jump_to": "model"}

print("Middleware XBRL definido.")


Middleware XBRL definido.


### Ensamblaje del agente final

`create_agent` reúne modelo, tools, prompt, esquema de respuesta, middleware y checkpointer. En este punto las piezas construidas de forma independiente pasan a formar un único sistema ejecutable.

El `InMemorySaver` permite mantener estado por `thread_id`. Durante la evaluación se utiliza un identificador distinto por pregunta para impedir contaminación accidental entre casos: una pregunta del golden set no debe beneficiarse del contexto de otra.


In [18]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agente_final = create_agent(
    model=MODELO,
    tools=HERRAMIENTAS,
    system_prompt=SYSTEM,
    response_format=RespuestaFinanciera,
    middleware=[
        ToolCallLimitMiddleware(run_limit=8),
        ModelCallLimitMiddleware(run_limit=10),
        verificar_cifras_contra_xbrl,
    ],
    checkpointer=InMemorySaver(),
)
print("Agente final creado.")


Agente final creado.


## 7. Interfaz pública `responder()` y observabilidad

La práctica necesita una función sencilla que pueda utilizarse tanto manualmente como durante las preguntas ciegas. `responder(pregunta)` encapsula toda la ejecución del agente y añade información operacional.

Además de la respuesta estructurada se registran:

- **latencia total**, medida con reloj monotónico;
- **tokens de entrada y salida**, cuando el proveedor los reporta;
- **coste estimado**, calculado a partir de una tarifa configurable;
- **trayectoria de mensajes y tools**, conservada en el resultado de LangGraph.

### Por qué medimos coste y latencia aquí

Estas métricas deben medirse alrededor de la ejecución real y no estimarse posteriormente. Colocarlas en la interfaz pública garantiza que todas las evaluaciones utilicen el mismo criterio. También permite comparar baseline y final de forma homogénea.

El coste se marca explícitamente como **estimado**, ya que depende de la tarifa vigente del proveedor y de que el modelo reporte correctamente el uso de tokens. Antes de la entrega definitiva conviene verificar la tarifa configurada si el proveedor la ha modificado.

La función auxiliar `mostrar_respuesta()` no interviene en la lógica del agente; únicamente ofrece una representación cómoda para inspección humana durante el desarrollo.


### Instrumentación de `responder()`

La implementación envuelve `agente_final.invoke` con medición temporal y extracción de uso de tokens. Los metadatos operativos se añaden al diccionario original en lugar de eliminar la trayectoria de LangGraph; esto es esencial porque el evaluador de uso de herramientas necesita inspeccionar posteriormente los mensajes.

La tarifa del modelo se mantiene como parámetro visible y documentado. No forma parte del razonamiento del agente: únicamente transforma tokens consumidos en una estimación monetaria comparable entre versiones.


In [19]:
# Precio configurable para poder reportar coste. Ajustar si cambia la tarifa del proveedor.
# USD por millón de tokens (entrada, salida). La evaluación conserva además los tokens brutos.
PRECIO_MODELO_USD_M = (0.75, 3.75)

def tokens_de(resultado) -> tuple[int, int]:
    entrada = salida = 0
    for mensaje in resultado.get("messages", []):
        uso = getattr(mensaje, "usage_metadata", None) or {}
        entrada += int(uso.get("input_tokens", 0) or 0)
        salida += int(uso.get("output_tokens", 0) or 0)
    return entrada, salida

def coste_estimado(resultado) -> float:
    entrada, salida = tokens_de(resultado)
    p_in, p_out = PRECIO_MODELO_USD_M
    return (entrada * p_in + salida * p_out) / 1e6

def responder(pregunta: str, thread_id: str | None = None) -> dict:
    """Ejecuta el agente final y añade latencia, tokens y coste estimado."""
    t0 = time.perf_counter()
    resultado = agente_final.invoke(
        {"messages": [{"role": "user", "content": pregunta}]},
        config={"configurable": {"thread_id": thread_id or "default"}},
    )
    latencia = time.perf_counter() - t0
    entrada, salida = tokens_de(resultado)
    return {
        **resultado,
        "latencia_s": latencia,
        "tokens_entrada": entrada,
        "tokens_salida": salida,
        "coste_usd": coste_estimado(resultado),
    }

def mostrar_respuesta(resultado: dict):
    r = resultado.get("structured_response")
    if r is None:
        print("Sin respuesta estructurada.")
        return
    print(r.respuesta)
    print(f"fuente={r.fuente} · ticker={r.ticker} · FY={r.ejercicio} · chunk={r.chunk_id}")
    print(f"latencia={resultado['latencia_s']:.2f}s · coste≈${resultado['coste_usd']:.5f}")

print("responder() listo.")


responder() listo.


### Smoke tests antes de una evaluación completa

Antes de lanzar 20 preguntas —y asumir su coste— se prueban dos rutas cualitativamente distintas: una pregunta numérica y una pregunta sobre riesgos. El objetivo no es medir rendimiento estadístico, sino detectar errores gruesos de integración: credenciales, routing, formato estructurado, tools y trazas.

También se imprimen las herramientas utilizadas. Una respuesta textual aparentemente correcta no basta para este smoke test; queremos observar que la pregunta numérica y la cualitativa recorren caminos coherentes con la arquitectura.


In [20]:
# Smoke test: una pregunta numérica y una cualitativa.
PRUEBAS = [
    "¿Cuál fue el revenue de NVIDIA en FY2024?",
    "¿Qué riesgos relacionados con IA menciona Microsoft en FY2025?",
]
for i, pregunta in enumerate(PRUEBAS, 1):
    print("\nPREGUNTA:", pregunta)
    r = responder(pregunta, thread_id=f"smoke-{i}")
    mostrar_respuesta(r)
    print("tools:", miax_s2.herramientas_usadas(r))



PREGUNTA: ¿Cuál fue el revenue de NVIDIA en FY2024?
El revenue (ingresos) de NVIDIA en el ejercicio fiscal 2024 (FY2024) fue de 60.922 millones de dólares (60.922.000.000 USD).
fuente=xbrl · ticker=NVDA · FY=2024 · chunk=None
latencia=2.97s · coste≈$0.00390
tools: ['get_xbrl_fact', 'RespuestaFinanciera']

PREGUNTA: ¿Qué riesgos relacionados con IA menciona Microsoft en FY2025?
En el Item 1A de su informe 10-K para el ejercicio fiscal 2025, Microsoft detalla varios riesgos clave relacionados con la inteligencia artificial:

1. Fallos en algoritmos, datos y generación de contenido: Algoritmos o metodologías de entrenamiento defectuosos, conjuntos de datos sesgados, insuficientes o inexactos, y generación de contenido perjudicial, ilegal, ofensivo o inexacto.
2. Responsabilidad legal, propiedad intelectual y privacidad: Posibles litigios o sanciones regulatorias derivados del uso de datos para el entrenamiento de modelos de IA y de los resultados (outputs) generados, incluyendo reclamaci

### 7.1 Caso demostrativo: por qué una comparativa justifica el agente

Una pregunta como “¿cómo cambió el beneficio neto entre dos ejercicios y qué lo explica?” no es una recuperación simple. Requiere al menos dos hechos estructurados —uno por ejercicio— y una explicación textual. Un RAG plano hace una sola recuperación y deja al LLM resolver lo que falte. El agente, en cambio, puede decidir que necesita varias llamadas.

La siguiente celda ejecuta una comparativa y muestra **la trayectoria completa**. Lo importante no es solo la respuesta final: queremos ver `get_xbrl_fact` para ambos ejercicios y `search_filings` para la explicación. Esa trayectoria es precisamente lo que después evalúa `uso_la_tool_correcta`.


In [21]:
pregunta_comp = "¿Cómo cambió el beneficio neto de Meta entre FY2024 y FY2025 y qué explica esa variación?"
print("PREGUNTA COMPARATIVA:\n", pregunta_comp)
try:
    r_comp = responder(pregunta_comp, thread_id="demo-comparativa")
    miax_s2.pretty_trace(r_comp)
    print("\nTools utilizadas:", miax_s2.herramientas_usadas(r_comp))
except Exception as e:
    print("La demo requiere acceso al modelo:", type(e).__name__, e)


PREGUNTA COMPARATIVA:
 ¿Cómo cambió el beneficio neto de Meta entre FY2024 y FY2025 y qué explica esa variación?
  1. list_available()
       -> AAPL (Apple Inc.): FY[2024, 2025], items ['1A', '7', '7A', '8'] AMZN (AMAZON COM INC): FY[2024, 2025], items ['1A', '7', '7A', '8'] GOOGL (Alphabet Inc.): FY[2024, 2025], items ['1A', '7', '7A', '8'] META (Meta Platforms…
  2. get_xbrl_fact(ticker='META', fiscal_year=2024, concept='NetIncomeLoss')
  3. get_xbrl_fact(fiscal_year=2025, concept='NetIncomeLoss', ticker='META')
       -> META FY2024 NetIncomeLoss = 62,360,000,000 USD (cierre 2024-12-31, 10-K)
       -> META FY2025 NetIncomeLoss = 60,458,000,000 USD (cierre 2025-12-31, 10-K)
  4. search_filings(k=5, ticker='META', query='net income decreased income taxes provision costs expenses', item='7', fiscal_year=2025)
       -> [META-2025-7-0020] META FY2025 Item 7 (similitud 0.028) of revenue	18			18			19 Research and development	29			27			29 Marketing and sales	6			7			9 General and adminis

### 7.2 Caso demostrativo del guardrail: concepto XBRL incorrecto

El guardrail se entiende mejor con una trampa real del corpus. El concepto de ingresos **no es universal entre compañías**. Si el modelo generaliza el concepto que usa Apple a Alphabet, la consulta puede no devolver nada. Sin control, un LLM puede completar una cifra plausible de memoria o de la prosa. Con el middleware, una cifra afirmada se contrasta contra XBRL y, si no cuadra, el modelo recibe una observación y tiene que corregirse.

Mostramos primero la consulta equivocada y después la correcta. Así queda visible que el problema no es “que el LLM no sepa finanzas”, sino que los esquemas XBRL reales tienen heterogeneidad semántica.


In [22]:
print("CONCEPTO GENERALIZADO INCORRECTAMENTE:")
print(get_xbrl_fact.invoke({"ticker":"GOOGL","fiscal_year":2025,
                            "concept":"RevenueFromContractWithCustomerExcludingAssessedTax"}))
print("\nCONCEPTO QUE FIGURA EN EL CORPUS PARA ALPHABET:")
print(get_xbrl_fact.invoke({"ticker":"GOOGL","fiscal_year":2025,"concept":"Revenues"}))

print("\nAhora ejecutamos el agente final para que la trayectoria muestre si verifica la cifra:")
try:
    r_guard = responder("¿Cuáles fueron los ingresos de Alphabet en FY2025?", thread_id="demo-guardrail")
    miax_s2.pretty_trace(r_guard)
except Exception as e:
    print("La demo requiere acceso al modelo:", type(e).__name__, e)


CONCEPTO GENERALIZADO INCORRECTAMENTE:
GOOGL no reportó 'RevenueFromContractWithCustomerExcludingAssessedTax' en FY2025. Conceptos disponibles: ['Assets', 'CashAndCashEquivalentsAtCarryingValue', 'EarningsPerShareBasic', 'EarningsPerShareDiluted', 'Liabilities', 'NetCashProvidedByUsedInOperatingActivities', 'NetIncomeLoss', 'OperatingIncomeLoss', 'ResearchAndDevelopmentExpense', 'Revenues', 'StockholdersEquity']

CONCEPTO QUE FIGURA EN EL CORPUS PARA ALPHABET:
GOOGL FY2025 Revenues = 402,836,000,000 USD (cierre 2025-12-31, 10-K)

Ahora ejecutamos el agente final para que la trayectoria muestre si verifica la cifra:
  1. list_available()
       -> AAPL (Apple Inc.): FY[2024, 2025], items ['1A', '7', '7A', '8'] AMZN (AMAZON COM INC): FY[2024, 2025], items ['1A', '7', '7A', '8'] GOOGL (Alphabet Inc.): FY[2024, 2025], items ['1A', '7', '7A', '8'] META (Meta Platforms…
  2. get_xbrl_fact(concept='Revenues', fiscal_year=2025, ticker='GOOGL')
       -> GOOGL FY2025 Revenues = 402,836,000,000 

## 8. Evaluadores automáticos: qué significa que una respuesta sea correcta

La evaluación se divide en tres dimensiones porque acertar el texto final no garantiza que el sistema haya seguido un procedimiento robusto. En esta práctica **el camino forma parte de la corrección**.

### 8.1 Evaluador de cita

Comprueba que la respuesta incluya un `chunk_id` válido y que la cita declarada aparezca realmente en el texto de ese fragmento. Esto detecta referencias inexistentes o citas que no corresponden al fragmento señalado.

La normalización elimina diferencias irrelevantes de espacios y mayúsculas. Se utiliza un prefijo de la cita para hacer la comprobación suficientemente robusta frente a pequeñas diferencias de formato, manteniendo la exigencia de que el contenido proceda del chunk indicado.

### 8.2 Evaluador numérico

Cuando el golden set contiene `cifra_esperada`, se compara el campo numérico de la salida estructurada contra ese valor mediante la tolerancia documentada. Una respuesta narrativa correcta pero con `cifra=None` falla en una pregunta numérica, porque el contrato de salida exige exponer la magnitud de forma estructurada.

### 8.3 Evaluador de trayectoria

Inspecciona las tools realmente utilizadas y verifica que incluyan las esperadas. Por ejemplo, acertar una cifra leyendo prosa no se considera equivalente a haber consultado XBRL. Este evaluador convierte una decisión arquitectónica —usar la fuente adecuada— en un criterio medible.

Los tres evaluadores se registran en un diccionario común para que `evaluar()` pueda aplicarlos de forma uniforme a cada pregunta.


### Implementación de los tres evaluadores

Cada función devuelve `True`, `False` o, cuando el criterio no aplica, `None`. Esta distinción es importante: una pregunta cualitativa sin `cifra_esperada` no debe contar como fallo numérico; simplemente queda fuera del denominador de esa métrica.

El evaluador de trayectoria trabaja con conjuntos porque algunas preguntas pueden requerir varias tools. Se comprueba que todas las herramientas esperadas estén contenidas entre las utilizadas, sin penalizar necesariamente llamadas adicionales; el número de llamadas se analiza por separado como métrica de eficiencia.


In [23]:
def cita_correcta(item: dict, resultado: dict) -> bool | None:
    r = resultado.get("structured_response")
    if r is None:
        return False if item.get("familia") == "extractiva" else None
    if not r.chunk_id:
        return False if item.get("ancla_texto") else None
    fragmento = POR_ID.get(r.chunk_id)
    if fragmento is None:
        return False
    if not r.cita:
        return False
    cita = miax_s2.normalizar(r.cita)[:120]
    texto = miax_s2.normalizar(fragmento.get("texto", ""))
    return bool(cita) and cita in texto

def cifra_coincide_xbrl(item: dict, resultado: dict) -> bool | None:
    r = resultado.get("structured_response")
    if r is None:
        return False if (item.get("cifra_esperada") is not None or item.get("cifras_esperadas") or item.get("espera_sin_datos")) else None

    # Casos fuera del corpus: lo correcto es abstenerse de dar una cifra.
    if item.get("espera_sin_datos"):
        return r.cifra is None and getattr(r, "fuente", None) == "ninguna"

    varias = item.get("cifras_esperadas") or {}
    if varias:
        # El esquema estructurado solo tiene un campo `cifra`; en una comparativa
        # comprobamos que los dos valores XBRL aparezcan en la respuesta textual.
        nums = miax_s2.extraer_cifras(r.respuesta or "")
        return all(any(miax_s2.cuadra(float(n), float(v), TOLERANCIA) for n in nums)
                   for v in varias.values())

    esperada = item.get("cifra_esperada")
    if esperada is None:
        return None
    if r.cifra is None:
        return False
    return miax_s2.cuadra(float(r.cifra), float(esperada), TOLERANCIA)

def uso_la_tool_correcta(item: dict, resultado: dict) -> bool:
    usadas = set(miax_s2.herramientas_usadas(resultado))
    esperadas = set(item.get("herramienta_esperada") or [])
    return esperadas.issubset(usadas)

EVALUADORES = {
    "cita": cita_correcta,
    "cifra": cifra_coincide_xbrl,
    "trayectoria": uso_la_tool_correcta,
}
print("Evaluadores:", list(EVALUADORES))


Evaluadores: ['cita', 'cifra', 'trayectoria']


## 9. Evaluación del golden set y generación de resultados

### Objetivo de `evaluar()`

`evaluar(ruta_jsonl)` constituye, junto con `responder()`, la interfaz principal de la entrega. Recorre un conjunto de preguntas, ejecuta el agente y genera una fila de resultados por caso. La evaluación está diseñada para continuar aunque una pregunta concreta produzca una excepción: el error se registra en esa fila en lugar de perder toda la ejecución.

Para cada pregunta se almacenan métricas de dos tipos:

**Calidad y procedimiento**
- corrección de la cita;
- coincidencia numérica;
- uso de las herramientas esperadas;
- `recall@5` cuando existe ancla de retrieval.

**Eficiencia**
- latencia;
- coste estimado;
- tokens de entrada y salida;
- número de llamadas a herramientas.

### Por qué se guardan resultados por pregunta

Una media global puede ocultar fallos sistemáticos. El CSV detallado permite analizar posteriormente qué familias fallan, qué compañías presentan más dificultad o qué preguntas disparan coste/latencia. A partir de ese fichero se construye una tabla resumen, pero se conserva siempre el nivel granular.

### Recall dentro de la evaluación

Para preguntas con `ancla_texto`, además de evaluar la respuesta del agente se ejecuta directamente el retriever final. Esto separa dos problemas: **¿recuperamos la evidencia?** y **¿el agente supo utilizarla?**. Si el recall falla, el problema está antes de la generación; si el recall acierta pero la respuesta falla, debemos investigar routing, síntesis o formato.

Los resultados se escriben a CSV para que las tablas de la memoria sean regenerables a partir del código entregado.


### Implementación de `evaluar()` y del resumen agregado

La función recorre el JSONL y utiliza un `thread_id` único por identificador de pregunta. Los resultados se acumulan como diccionarios y finalmente se convierten en DataFrame, una representación conveniente para inspección, agregación y exportación a CSV.

`resumir_resultados` calcula medias únicamente sobre valores no nulos. Esto evita, por ejemplo, que las preguntas extractivas se interpreten como fallos numéricos. Las métricas se mantienen en escala 0–1 y solo se formatean como porcentaje en la visualización, preservando un formato numérico reutilizable en análisis posteriores.


In [24]:
def evaluar(ruta_jsonl: str | Path, salida_csv: str | Path | None = None) -> pd.DataFrame:
    """Evalúa `responder()` sobre un JSONL compatible con el esquema del golden set."""
    ruta = Path(ruta_jsonl)
    preguntas = [json.loads(l) for l in ruta.open(encoding="utf-8") if l.strip()]
    filas = []
    for i, item in enumerate(preguntas, 1):
        print(f"[{i:02d}/{len(preguntas):02d}] {item['id']}", end="\r")
        fila = {"id": item["id"], "familia": item["familia"], "ticker": item["ticker"]}
        try:
            r = responder(item["pregunta"], thread_id=f"eval-{item['id']}")
            fila.update({
                "latencia_s": r["latencia_s"],
                "coste_usd": r["coste_usd"],
                "tokens_entrada": r["tokens_entrada"],
                "tokens_salida": r["tokens_salida"],
                "llamadas": len(miax_s2.herramientas_usadas(r)),
            })
            for nombre, fn in EVALUADORES.items():
                fila[nombre] = fn(item, r)
            if item.get("ancla_texto"):
                q = reescribir_consulta(item["pregunta"])
                rec = buscar_hibrido(q, item.get("ticker"), item.get("fiscal_year"),
                                     item.get("item_esperado"), 5)
                fila["recall@5"] = miax_s2.acierta(item, rec)
            else:
                fila["recall@5"] = None
            sr = r.get("structured_response")
            fila["respuesta"] = sr.respuesta if sr else None
        except Exception as e:
            fila["error"] = f"{type(e).__name__}: {e}"
        filas.append(fila)

    tabla = pd.DataFrame(filas)
    if salida_csv is not None:
        tabla.to_csv(salida_csv, index=False)
    print(" " * 50, end="\r")
    return tabla

def resumir_resultados(tabla: pd.DataFrame, version: str = "final") -> pd.DataFrame:
    def media_bool(col):
        if col not in tabla:
            return float("nan")
        x = tabla[col].dropna()
        return float(x.astype(float).mean()) if len(x) else float("nan")
    return pd.DataFrame([{
        "versión": version,
        "cita": media_bool("cita"),
        "cifra": media_bool("cifra"),
        "trayectoria": media_bool("trayectoria"),
        "recall@5": media_bool("recall@5"),
        "coste medio (USD)": tabla.get("coste_usd", pd.Series(dtype=float)).mean(),
        "latencia media (s)": tabla.get("latencia_s", pd.Series(dtype=float)).mean(),
        "llamadas/pregunta": tabla.get("llamadas", pd.Series(dtype=float)).mean(),
    }])

print("evaluar() y resumir_resultados() listos.")


evaluar() y resumir_resultados() listos.


### Ejecución de la evaluación final

Si el golden set existe, se ejecuta el sistema final completo y se guarda `resultados_final.csv` en Drive. La tabla mostrada resume el comportamiento, pero el CSV detallado es el artefacto principal para análisis y reproducibilidad. Si falta el fichero, se emite un mensaje claro y se evita una ejecución parcial que pudiera confundirse con resultados definitivos.


In [25]:
if RUTA_GOLDEN.is_file() and not validar_golden(golden, exigir_anclas=True):
    RESULTADOS_FINAL = DATA_DIR / "resultados_final.csv"
    tabla_final = evaluar(RUTA_GOLDEN, RESULTADOS_FINAL)
    resumen_final = resumir_resultados(tabla_final, "final")
    display(resumen_final.style.format({
        "cita": "{:.1%}", "cifra": "{:.1%}", "trayectoria": "{:.1%}",
        "recall@5": "{:.1%}", "coste medio (USD)": "${:.5f}",
        "latencia media (s)": "{:.2f}", "llamadas/pregunta": "{:.2f}",
    }))
    print("Resultados guardados en:", RESULTADOS_FINAL)
else:
    print("Evaluación final pendiente: cierra primero las anclas textuales del golden set.")


Evaluación final pendiente: cierra primero las anclas textuales del golden set.


## 10. Comparación experimental: baseline frente a sistema final

### Por qué esta comparación es imprescindible

Añadir componentes no demuestra que el sistema haya mejorado. El retrieval híbrido, la reescritura o los guardrails también introducen complejidad y pueden aumentar latencia y coste. Por ello se evalúan **baseline y final sobre exactamente el mismo golden set**.

### Definición del baseline

El baseline conserva la búsqueda densa original de la sesión 1 y las cuatro herramientas básicas, pero no incorpora el retrieval híbrido ni el middleware de verificación XBRL. De esta forma representa de manera razonable el sistema antes de las mejoras de robustez.

### Definición del sistema final

La versión final incorpora:

- filtrado explícito por metadatos;
- combinación BM25 + denso mediante RRF;
- reescritura de consultas;
- límites de ejecución;
- verificación programática de cifras;
- las mismas interfaces de salida y evaluación.

### Qué se compara

La tabla conjunta presenta métricas de calidad (`cita`, `cifra`, `trayectoria`, `recall@5`) junto a métricas operativas (`coste medio`, `latencia media`, `llamadas/pregunta`). Esto permite interpretar cualquier mejora con su coste asociado.

Por ejemplo, un aumento de recall acompañado de más latencia puede ser un compromiso aceptable o no dependiendo de su magnitud. El objetivo del experimento no es demostrar a priori que todas las modificaciones son positivas, sino **medir qué aportan realmente**. Si una mejora razonable no mueve la métrica, ese resultado también es informativo.

Los CSV `resultados_baseline.csv` y `resultados_final.csv` se guardan por separado para preservar la evidencia experimental y permitir reconstruir las tablas sin volver a ejecutar inmediatamente todas las llamadas al modelo.


### Construcción y evaluación del baseline comparable

Para que la comparación sea justa, el baseline utiliza el mismo modelo, esquema de salida, herramientas deterministas y mecanismo de medición; la diferencia principal se concentra en el retrieval y en la ausencia de los guardrails añadidos al sistema final.

Se crean funciones paralelas `responder_baseline` y `evaluar_con` para poder aplicar exactamente el mismo protocolo a ambas versiones. Después se guardan ambos CSV y se concatenan los resúmenes en una única tabla. La comparación debe interpretarse conjuntamente: calidad, coste, latencia y número de llamadas.


In [26]:
from langchain.agents import create_agent

# Baseline reproducible usando las tools de la sesión 1.
@tool
def search_filings_baseline(query: str, ticker: str | None = None,
                            fiscal_year: int | None = None,
                            item: str | None = None, k: int = 5) -> str:
    """Busca fragmentos relevantes mediante la búsqueda densa original."""
    return miax_s1.formatear_fragmentos(
        buscar_denso_baseline(query, ticker, fiscal_year, item, k)
    )

TOOLS_BASE = [list_available, get_xbrl_fact, search_filings_baseline, read_section]
agente_baseline = create_agent(
    model=MODELO,
    tools=TOOLS_BASE,
    system_prompt=SYSTEM,
    response_format=RespuestaFinanciera,
    checkpointer=InMemorySaver(),
)

def responder_baseline(pregunta: str, thread_id: str | None = None) -> dict:
    t0 = time.perf_counter()
    r = agente_baseline.invoke(
        {"messages": [{"role": "user", "content": pregunta}]},
        config={"configurable": {"thread_id": thread_id or "baseline"}},
    )
    dt = time.perf_counter() - t0
    tin, tout = tokens_de(r)
    return {**r, "latencia_s": dt, "tokens_entrada": tin, "tokens_salida": tout,
            "coste_usd": coste_estimado(r)}

def evaluar_con(preguntas: list[dict], fn_responder, etiqueta: str) -> pd.DataFrame:
    filas = []
    for i, item in enumerate(preguntas, 1):
        print(f"{etiqueta} [{i:02d}/{len(preguntas):02d}] {item['id']}", end="\r")
        fila = {"id": item["id"], "familia": item["familia"], "ticker": item["ticker"]}
        try:
            r = fn_responder(item["pregunta"], thread_id=f"{etiqueta}-{item['id']}")
            fila.update({"latencia_s": r["latencia_s"], "coste_usd": r["coste_usd"],
                         "llamadas": len(miax_s2.herramientas_usadas(r))})
            for nombre, ev in EVALUADORES.items():
                fila[nombre] = ev(item, r)
            if item.get("ancla_texto"):
                buscador = (buscar_denso_baseline if etiqueta == "baseline" else buscar_hibrido)
                q = item["pregunta"] if etiqueta == "baseline" else reescribir_consulta(item["pregunta"])
                rec = buscador(q, item.get("ticker"), item.get("fiscal_year"), item.get("item_esperado"), 5)
                fila["recall@5"] = miax_s2.acierta(item, rec)
            else:
                fila["recall@5"] = None
        except Exception as e:
            fila["error"] = f"{type(e).__name__}: {e}"
        filas.append(fila)
    print(" " * 60, end="\r")
    return pd.DataFrame(filas)

if golden and not validar_golden(golden, exigir_anclas=True):
    t_base = evaluar_con(golden, responder_baseline, "baseline")
    t_final = evaluar_con(golden, responder, "final")
    t_base.to_csv(DATA_DIR / "resultados_baseline.csv", index=False)
    t_final.to_csv(DATA_DIR / "resultados_final.csv", index=False)
    comparacion = pd.concat([
        resumir_resultados(t_base, "baseline"),
        resumir_resultados(t_final, "final"),
    ], ignore_index=True)
    display(comparacion.style.format({
        "cita": "{:.1%}", "cifra": "{:.1%}", "trayectoria": "{:.1%}",
        "recall@5": "{:.1%}", "coste medio (USD)": "${:.5f}",
        "latencia media (s)": "{:.2f}", "llamadas/pregunta": "{:.2f}",
    }))
else:
    print("Comparación pendiente hasta cerrar y validar las anclas del golden set.")


Comparación pendiente hasta cerrar y validar las anclas del golden set.


### 10.1 Comparación por pregunta, no solo por promedio

Un promedio puede ocultar regresiones. Por ejemplo, el sistema final puede mejorar mucho las extractivas pero empeorar una pregunta sencilla por añadir una reescritura innecesaria. Por eso, además del resumen global, construimos una tabla **pregunta a pregunta** con el delta de cita, cifra, trayectoria, recall, coste y latencia.

Esta es una de las tablas más útiles para la defensa: permite señalar un caso concreto y decir “aquí el filtro corrigió el ejercicio”, “aquí BM25 no aportó” o “aquí el guardrail añadió una llamada pero evitó una cifra no verificada”.


In [27]:
if golden and 't_base' in globals() and 't_final' in globals():
    cols=[c for c in ["id","familia","cita","cifra","trayectoria","recall@5","coste_usd","latencia_s","llamadas"] if c in t_base.columns and c in t_final.columns]
    detalle=t_base[cols].merge(t_final[cols],on=["id","familia"],suffixes=("_base","_final"))
    for c in ["coste_usd","latencia_s","llamadas"]:
        if c+"_base" in detalle and c+"_final" in detalle:
            detalle["delta_"+c]=detalle[c+"_final"]-detalle[c+"_base"]
    display(detalle)

    # Resumen por familia: evita que una familia numerosa domine el promedio global.
    filas=[]
    for fam in sorted(detalle["familia"].unique()):
        d=detalle[detalle.familia==fam]
        fila={"familia":fam,"n":len(d)}
        for met in ["cita","cifra","trayectoria","recall@5"]:
            for ver in ["base","final"]:
                col=f"{met}_{ver}"
                if col in d:
                    x=d[col].dropna(); fila[col]=x.astype(float).mean() if len(x) else np.nan
        filas.append(fila)
    display(pd.DataFrame(filas).style.format({c:"{:.1%}" for c in pd.DataFrame(filas).columns if c not in {"familia","n"}}))
else:
    print("Esta tabla se genera automáticamente después de ejecutar baseline y final sobre el golden.")


Esta tabla se genera automáticamente después de ejecutar baseline y final sobre el golden.


### 10.2 Lectura automática de mejoras y regresiones

La siguiente celda no sustituye el análisis humano; simplemente localiza los casos interesantes. Marca preguntas en las que el sistema final pasa de fallo a acierto y también aquellas en las que ocurre lo contrario. De este modo no seleccionamos únicamente ejemplos favorables: **las regresiones también forman parte del resultado experimental**.


In [28]:
if 'detalle' in globals():
    for met in ["cita","cifra","trayectoria","recall@5"]:
        b,f=f"{met}_base",f"{met}_final"
        if b not in detalle or f not in detalle: continue
        mejora=detalle[(detalle[b]==False)&(detalle[f]==True)]
        empeora=detalle[(detalle[b]==True)&(detalle[f]==False)]
        print(f"\n{met.upper()}: +{len(mejora)} mejoras / -{len(empeora)} regresiones")
        if len(mejora): print("  mejoran:", mejora['id'].tolist())
        if len(empeora): print("  empeoran:", empeora['id'].tolist())


## 11. Verificación final de contrato y preparación de la entrega

Antes de considerar terminado el notebook se ejecutan comprobaciones mínimas sobre los elementos que utilizará el evaluador. Estas aserciones no sustituyen a los tests funcionales, pero detectan errores de integración frecuentes: renombrar accidentalmente una tool, olvidar una interfaz pública o perder uno de los evaluadores.

Se verifica que:

- `responder` y `evaluar` existen y son invocables;
- están registrados exactamente los tres evaluadores previstos;
- las cuatro tools conservan los nombres contractuales;
- la función de tolerancia numérica acepta un redondeo pequeño y rechaza una desviación grande;
- el diccionario de chunks mantiene una entrada por fragmento.

La finalidad de esta última sección es aproximarnos al escenario del día de evaluación: **repositorio limpio, ejecución completa y funciones públicas disponibles sin editar código**.


### Tests de integración mínimos

Las aserciones siguientes actúan como una última barrera frente a cambios accidentales antes de subir el repositorio. En particular, la lista exacta de nombres de tools protege el contrato externo de la práctica. También se realizan dos comprobaciones sencillas sobre la tolerancia numérica para verificar que su comportamiento básico coincide con lo esperado.


In [29]:
assert callable(responder)
assert callable(evaluar)
assert set(EVALUADORES) == {"cita", "cifra", "trayectoria"}
assert [t.name for t in HERRAMIENTAS] == [
    "list_available", "get_xbrl_fact", "search_filings", "read_section"
]
assert miax_s2.cuadra(100.0, 100.4, 0.01)
assert not miax_s2.cuadra(100.0, 150.0, 0.01)
assert len(chunks) == len(POR_ID)
print("✓ Tools: OK")
print("✓ responder(): OK")
print("✓ evaluar(): OK")
print("✓ 3 evaluadores: OK")
print("✓ guardrail XBRL: definido")
print("✓ retrieval híbrido: definido")
print("\nNotebook preparado para la ejecución final en Colab.")


✓ Tools: OK
✓ responder(): OK
✓ evaluar(): OK
✓ 3 evaluadores: OK
✓ guardrail XBRL: definido
✓ retrieval híbrido: definido

Notebook preparado para la ejecución final en Colab.


## 12. Notas de reproducibilidad, limitaciones y checklist de entrega

### Ficheros que deben acompañar al notebook

El repositorio debe contener el código necesario para reconstruir el sistema, el `golden_set.jsonl` propio y los resultados regenerables. Los auxiliares `miax_s1.py` y `miax_s2.py` se conservan sin modificar. Los ZIP del corpus pueden permanecer fuera del repositorio si esa es la organización acordada para la práctica, pero el README debe explicar de forma inequívoca dónde deben situarse.

### Credenciales

No se almacena ninguna API key en el notebook. La clave se obtiene de **Colab Secrets**. Antes de subir el repositorio debe revisarse también el historial de outputs por si alguna ejecución anterior hubiese impreso accidentalmente información sensible.

### Dependencia de servicios externos

Aunque el código sea reproducible, la respuesta de un LLM remoto no es completamente determinista ni está bajo nuestro control. Pueden existir cambios de proveedor, disponibilidad o tarifas. Por eso se separan las métricas deterministas —por ejemplo, la validación XBRL— de las métricas dependientes de una ejecución del modelo, y se guardan los CSV obtenidos.

### Limitaciones conocidas

1. La reescritura añade una llamada adicional al modelo y, por tanto, coste y latencia. Debe justificarse con la mejora observada en recall.
2. BM25 opera sobre el troceado proporcionado. Un cambio de chunking requeriría reconstruir tanto el índice léxico como el vectorial.
3. La verificación numérica comprueba compatibilidad con hechos XBRL, pero no sustituye el razonamiento necesario para preguntas derivadas o comparativas.
4. `read_section` puede ser costosa en tokens y debe seguir considerándose un fallback.
5. Las métricas sobre el golden set propio miden comportamiento en ese conjunto; las preguntas ciegas son la prueba de generalización.

### Checklist antes de entregar

- Ejecutar **Entorno de ejecución → Reiniciar sesión y ejecutar todo** en un Colab limpio.
- Confirmar que Drive contiene los dos ZIP, `miax_s1.py`, `miax_s2.py` y `golden_set.jsonl`.
- Verificar que el golden set contiene 20 preguntas y al menos 6 comparativas.
- Confirmar que se generan `resultados_baseline.csv` y `resultados_final.csv`.
- Revisar la tabla baseline vs. final y trasladar sus valores reales al informe/presentación.
- Comprobar que `responder()` y `evaluar()` funcionan sin editar celdas.
- Verificar que ninguna API key aparece en código, outputs o repositorio.
- Ejecutar varias preguntas manuales y revisar la trayectoria de tools, no solo la respuesta final.

### Conclusión técnica

La solución final combina recuperación semántica y léxica, datos financieros estructurados y control programático alrededor del LLM. El diseño busca que el modelo se utilice donde aporta valor —interpretación, routing y síntesis— y que las operaciones que pueden resolverse de forma determinista —consulta de cifras, validación y evaluación— permanezcan fuera de la generación libre. Esta separación es la principal decisión arquitectónica del trabajo y permite construir un agente más trazable y evaluable que un RAG monolítico basado únicamente en similitud semántica.


## 13. Qué debe observarse al ejecutar el notebook completo

La ejecución final debe dejar una historia experimental visible, no solo código:

1. Las tools deterministas muestran por qué una cifra se consulta en XBRL y por qué la ausencia de un concepto se trata explícitamente.
2. El golden set aparece como un fichero JSONL independiente y validado.
3. La ablación de retrieval enseña el efecto incremental de metadatos, híbrido y reescritura.
4. Al menos un ejemplo imprime los top-5 antes/después para ver el error corregido.
5. Una pregunta comparativa imprime la trayectoria multi-tool del agente.
6. El caso de Alphabet enseña el riesgo de generalizar conceptos XBRL entre compañías.
7. Baseline y final se comparan globalmente, por familia y por pregunta.
8. Se muestran también las regresiones, no solo los éxitos.
9. Coste, latencia y número de llamadas acompañan a la calidad: una mejora más cara tiene que justificar ese coste.

Con esto, la conclusión de la práctica deja de ser “hemos construido un agente” y pasa a ser una afirmación mucho más defendible: **qué fallaba, qué cambio introdujimos, qué evidencia muestra que el cambio ayudó —o no— y cuánto costó conseguirlo**.
